# Module 5 Lab: One Line or Many?
## 6.3710x — Probability and Statistical Data Analysis

### Where We Left Off

In Module 4, PCA revealed the hidden variable in the crab dataset: **species**. What first looked like one broad cloud was better understood as two populations. You also used AIC to formalize the idea that a richer model can be justified when the gain in fit outweighs the penalty for complexity.

But PCA is an unsupervised tool. It told you about variation, not about **prediction**.

### What's Changed

In this lab, we move from covariance structure to **regression**.

We'll ask a different kind of scientific question:

> Can one crab measurement be predicted from another by a single common linear relationship, or do sex and species change that relationship?

This is the capstone move of the course: you will build a model, test its coefficients, diagnose its failures, and then decide whether a richer model is warranted.

### The Plan

We'll use the crab measurements to study the relationship between frontal lobe size `FL` and carapace length `CL`.

Why this pair?

Earlier in the course, ratios involving `FL` and `CL` hinted that something interesting was happening. Now we'll revisit that same scientific story through the lens of regression.

We'll proceed in stages:

1. fit a **pooled simple regression** and quantify the slope
2. inspect residuals and discover what the pooled model hides
3. add **species** and **sex** as predictors
4. test whether the richer model is statistically justified
5. allow **species-specific slopes** via interaction terms
6. compare models from the **prediction** viewpoint
7. compress the data into a **one-dimensional score** and test it

### What's New Conceptually

- **Simple linear regression** as a best linear predictor
- **Multiple regression** with indicator variables
- **Confidence intervals** and **$t$-tests** for regression coefficients
- **Nested-model $F$-tests**
- **Interaction terms** and what they mean scientifically
- **Residual diagnostics**
- **Cross-validation** for comparing predictive performance
- **Pooling pitfalls**: significance is not the same as adequacy, and association is not causation


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, f, probplot, mannwhitneyu

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

np.random.seed(42)

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    base_url = ("https://raw.githubusercontent.com/"
                "codey-m/prob-stats-labs/main/")
    url = base_url + filename
    print(f"Loading from GitHub: {filename}")
    return pd.read_csv(url)

## Part 1: From Clouds to Lines

Let's return to the crab dataset. This time, instead of asking about overall covariance structure, we'll assign one variable the role of **predictor** and another the role of **response**.

We'll use:

- predictor: `CL` (carapace length)
- response: `FL` (frontal lobe size)

This is a natural pair because earlier in the course, the ratio `FL / CL` hinted at hidden structure.


In [ ]:
crabs = load_data("crabs.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))
crabs.head()

In [ ]:
measurement_cols = ["FL", "RW", "CL", "CW", "BD"]
crabs[measurement_cols].describe().round(2)

### TODO 1a

Extract:

- the predictor array $x$ from column `CL`
- the response array $y$ from column `FL`

Also record the sample size $n$.


In [ ]:
# TODO 1a: Extract predictor and response
x = ...
y = ...
n = ...

print(f"n = {n}")
print(f"x shape = {x.shape}")
print(f"y shape = {y.shape}")

### TODO 1b

Build the pooled design matrix for simple regression:

$$
X_\text{pooled} =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix}
$$

Use `np.ones(n)` to create the intercept column and `np.column_stack` to combine the columns.


In [ ]:
# TODO 1b: Build pooled design matrix
X_pooled = ...

print(f"Shape of X_pooled: {X_pooled.shape}")

In [ ]:
# Provided: first look at the regression problem
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# Left: colored by sex
for sex, color, marker in [("M", "steelblue", "o"),
                           ("F", "coral", "s")]:
    mask = crabs["sex"] == sex
    axes[0].scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                    c=color, marker=marker, s=35,
                    alpha=0.7, edgecolor="white", linewidth=0.3,
                    label=sex)
axes[0].set_title("FL vs CL — Colored by Sex",
                  fontsize=13, fontweight="bold")
axes[0].set_xlabel("CL")
axes[0].set_ylabel("FL")
axes[0].legend(fontsize=10)

# Right: colored by species
for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    axes[1].scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                    c=color, s=35, alpha=0.7,
                    edgecolor="white", linewidth=0.3,
                    label=label)
axes[1].set_title("FL vs CL — Colored by Species",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("CL")
axes[1].set_ylabel("FL")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

Look at both plots. There is clearly a strong linear trend in the pooled data.

But pay attention to the right panel. Do the two species seem to follow the *same* line, or could their relationships differ?

Hold that thought. Let's first see what a single pooled line looks like.


## Part 2: Pooled Simple Regression

We begin with the simplest model:

$$
Y = \beta_0 + \beta_1 X + Z
$$

This says that all crabs, regardless of species or sex, share one common linear relationship between `CL` and `FL`.


### TODO 2a

Compute the pooled OLS coefficients using the matrix formula:

$$
\hat\beta = (X^T X)^{-1} X^T y
$$

Use `np.linalg.solve(XtX, Xty)` rather than explicitly computing the inverse — it is more numerically stable.


In [ ]:
# TODO 2a: OLS via the normal equations
XtX = ...
Xty = ...

beta_pooled = ...

beta0_pooled = beta_pooled[0]
beta1_pooled = beta_pooled[1]

print(f"Pooled intercept: {beta0_pooled:.4f}")
print(f"Pooled slope:     {beta1_pooled:.4f}")

### TODO 2b

Compute the fitted values, residuals, and goodness of fit:

$$
\hat{y} = X \hat{\beta}, \qquad
e = y - \hat{y}
$$

$$
\mathrm{RSS} = \sum e_i^2, \qquad
\mathrm{TSS} = \sum (y_i - \bar{y})^2, \qquad
R^2 = 1 - \frac{\mathrm{RSS}}{\mathrm{TSS}}
$$


In [ ]:
# TODO 2b: Fit summaries for pooled model
y_hat_pooled = ...
residuals_pooled = ...

RSS_pooled = ...
TSS_pooled = ...
R2_pooled = ...

print(f"RSS (pooled): {RSS_pooled:.4f}")
print(f"TSS:          {TSS_pooled:.4f}")
print(f"R^2 (pooled): {R2_pooled:.4f}")

### TODO 2c

Now let's do inference on the pooled slope.

The estimated variance of the residuals is:

$$
\hat\sigma^2 = \frac{\mathrm{RSS}}{n - p}
$$

where $p$ is the number of parameters (2 for simple regression: intercept + slope).

The covariance matrix of $\hat\beta$ is:

$$
\widehat{\mathrm{Cov}}(\hat\beta) =
\hat\sigma^2 (X^T X)^{-1}
$$

The standard error of $\hat\beta_1$ is the square root of the $(1,1)$ entry (second diagonal element) of this matrix.

Compute:

- $\hat\sigma^2$
- the standard error of the slope
- the $t$-statistic: $t = \hat\beta_1 / \mathrm{SE}(\hat\beta_1)$
- the two-sided p-value using `t.sf(|t|, df) * 2` from `scipy.stats`


In [ ]:
# TODO 2c: Inference for the pooled slope
p_pooled = X_pooled.shape[1]
df_pooled = ...

sigma2_hat_pooled = ...
cov_beta_pooled = ...
se_beta1_pooled = ...

t_stat_beta1_pooled = ...
p_value_beta1_pooled = ...

print(f"SE(slope):     {se_beta1_pooled:.5f}")
print(f"t-statistic:   {t_stat_beta1_pooled:.3f}")
print(f"p-value:       {p_value_beta1_pooled:.3e}")

In [ ]:
# Provided: pooled regression line
x_grid = np.linspace(x.min(), x.max(), 200)
y_grid_pooled = beta0_pooled + beta1_pooled * x_grid

fig, ax = plt.subplots(figsize=(8, 6))

for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
               c=color, s=35, alpha=0.65,
               edgecolor="white", linewidth=0.3,
               label=label)

ax.plot(x_grid, y_grid_pooled, color="black", linewidth=2.5,
        label="Pooled regression line")

ax.set_xlabel("CL")
ax.set_ylabel("FL")
ax.set_title("Pooled Simple Regression",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The pooled $R^2$ is high, and the slope is overwhelmingly significant. At first glance, this model seems to work.

But a highly significant slope only tells you there is evidence of **some linear association** between `CL` and `FL`.

It does **not** tell you whether one common line is an adequate model for all crabs.

That distinction — significant vs. adequate — is the heart of the rest of this lab.

---


### OLS Helper Function

You've now computed OLS estimates, residuals, $R^2$, standard errors, and $t$-statistics by hand. For the rest of the lab, we'll use a helper that bundles these steps so you can focus on interpreting the results rather than re-deriving them.

Read through the function below to confirm it matches what you just did.


In [ ]:
def fit_ols(X, y):
    """Fit OLS using the normal equations and return summaries.

    Parameters
    ----------
    X : array, shape (n, p)
        Design matrix (must include an intercept column).
    y : array, shape (n,)
        Response vector.

    Returns
    -------
    dict with keys:
        beta, y_hat, residuals, RSS, TSS, R2,
        sigma2_hat, cov_beta, se_beta, n, p
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    n, p = X.shape
    beta = np.linalg.solve(X.T @ X, X.T @ y)
    y_hat = X @ beta
    residuals = y - y_hat

    RSS = np.sum(residuals**2)
    TSS = np.sum((y - y.mean())**2)
    R2 = 1 - RSS / TSS

    sigma2_hat = RSS / (n - p)
    cov_beta = sigma2_hat * np.linalg.inv(X.T @ X)
    se_beta = np.sqrt(np.diag(cov_beta))

    return {
        "beta": beta,
        "y_hat": y_hat,
        "residuals": residuals,
        "RSS": RSS,
        "TSS": TSS,
        "R2": R2,
        "sigma2_hat": sigma2_hat,
        "cov_beta": cov_beta,
        "se_beta": se_beta,
        "n": n,
        "p": p,
    }

In [ ]:
# Checkpoint: verify the helper matches your manual computation
_check = fit_ols(X_pooled, y)
assert np.allclose(beta_pooled, _check["beta"]), \
    "Helper coefficients do not match your manual computation."
assert np.isclose(R2_pooled, _check["R2"]), \
    "Helper R^2 does not match your manual computation."
print("✓ fit_ols agrees with your manual results.")

## Part 3: Diagnostics — What Pooling Hides

Residuals tell you what the model failed to explain.

If one common line really were adequate, the residuals should look like noise — centered around zero, with no systematic structure.

Before looking at the plots below, think about what you expect. The pooled $R^2$ is high and the slope is significant. Do you think the residuals will look like structureless noise?


In [ ]:
# Provided: residual plot colored by species
fig, ax = plt.subplots(figsize=(8, 5))

for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    ax.scatter(crabs.loc[mask, "CL"], residuals_pooled[mask],
               c=color, s=35, alpha=0.7,
               edgecolor="white", linewidth=0.3,
               label=label)

ax.axhline(0, color="black", linewidth=1.5, linestyle="--")
ax.set_xlabel("CL")
ax.set_ylabel("Residual")
ax.set_title("Residuals from the Pooled Model — Colored by Species",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Provided: QQ plot of pooled residuals
fig, ax = plt.subplots(figsize=(6, 5))
probplot(residuals_pooled, dist="norm", plot=ax)
ax.set_title("QQ Plot of Pooled Residuals",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### TODO 3a

The residual plot should look striking. To quantify what you see, compute the mean pooled residual within each species.

If the pooled model were adequate, these group means should both be near zero.


In [ ]:
# TODO 3a: Mean residual by species
resid_df = crabs[["sp", "sex"]].copy()
resid_df["resid_pooled"] = residuals_pooled

mean_resid_by_species = ...

print("Mean residual by species:")
print(mean_resid_by_species.round(4))

In [ ]:
# Provided: boxplots of residuals by group
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.boxplot(data=resid_df, x="sp", y="resid_pooled",
            palette=["dodgerblue", "darkorange"], ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Residuals by Species",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Species")
axes[0].set_ylabel("Pooled residual")

sns.boxplot(data=resid_df, x="sex", y="resid_pooled",
            palette=["coral", "steelblue"], ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Residuals by Sex",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Pooled residual")

plt.tight_layout()
plt.show()

This is the key lesson of this lab.

The pooled slope was overwhelmingly significant. The $R^2$ was high. But the residuals reveal systematic bias: one species is consistently over-predicted, the other under-predicted. The single line is **significant but not adequate**.

A p-value tests whether a coefficient is zero. It does not test whether the model is correctly specified.

The next step is to let the model account for group structure directly.

---


## Part 4: Multiple Regression with Group Indicators

The pooled model forced all crabs onto one common line.

Now we'll allow the line to **shift** depending on species and sex, while keeping one common slope.

The additive model:

$$
Y = \beta_0 + \beta_1 X + \beta_2 I_{\text{Blue}}
+ \beta_3 I_{\text{Female}} + Z
$$

where:

- $I_{\text{Blue}} = 1$ for Blue crabs, 0 otherwise
- $I_{\text{Female}} = 1$ for Female crabs, 0 otherwise

Interpretation: $\beta_2$ is how much higher (or lower) the regression line sits for Blue crabs compared to Orange crabs, **holding `CL` and sex fixed**. Similarly for $\beta_3$.


### TODO 4a

Create the indicator variables and build the additive design matrix. Each row of $X_\text{add}$ should be:

$$
[1, \; x_i, \; I_{\text{Blue},i}, \; I_{\text{Female},i}]
$$

Use comparisons like `(crabs["sp"] == "B").astype(float)` to create indicator columns.


In [ ]:
# TODO 4a: Group indicators and additive design matrix
is_blue = ...
is_female = ...

X_add = ...

print(f"Shape of X_add: {X_add.shape}")

### TODO 4b

Fit the additive model using `fit_ols` and extract the coefficient estimates, standard errors, and $R^2$.


In [ ]:
# TODO 4b: Fit additive model
model_add = ...
beta_add = ...
se_add = ...
R2_add = ...

print("Additive-model coefficients:")
labels_add = ["Intercept", "CL", "Blue", "Female"]
for name, coef, se in zip(labels_add, beta_add, se_add):
    print(f"  {name:>10s}: {coef:+.4f}  (SE = {se:.4f})")
print(f"\nR^2 (additive): {R2_add:.4f}")

### TODO 4c

Now test whether species and sex are jointly needed.

The **nested-model $F$-test** compares a reduced model (pooled) to a full model (additive) by asking: does the reduction in RSS from adding the group indicators justify the extra parameters?

$$
F = \frac{(\mathrm{RSS}_{\text{reduced}} -
\mathrm{RSS}_{\text{full}}) \;/\; q}
{\mathrm{RSS}_{\text{full}} \;/\; (n - p_{\text{full}})}
$$

where $q = p_{\text{full}} - p_{\text{reduced}}$ is the number of added parameters.

Under $H_0$: the extra coefficients are all zero, $F$ follows an $F(q,\; n - p_{\text{full}})$ distribution.

Compute $F$ and its p-value using `f.sf` from `scipy.stats`.


In [ ]:
# TODO 4c: Joint F-test — pooled vs additive
RSS_add = ...

q_groups = ...
F_groups = ...
p_value_groups = ...

print(f"F statistic (pooled vs additive): {F_groups:.3f}")
print(f"p-value:                          {p_value_groups:.3e}")

In [ ]:
# Provided: additive model fit visualized by species and sex
x_grid = np.linspace(x.min(), x.max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for j, sex_label in enumerate(["M", "F"]):
    ax = axes[j]
    sex_val = 1 if sex_label == "F" else 0

    for sp, color, label in [("O", "darkorange", "Orange"),
                             ("B", "dodgerblue", "Blue")]:
        sp_val = 1 if sp == "B" else 0
        mask = (crabs["sex"] == sex_label) & (crabs["sp"] == sp)

        ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                   c=color, s=35, alpha=0.65,
                   edgecolor="white", linewidth=0.3)

        X_grid = np.column_stack([
            np.ones_like(x_grid),
            x_grid,
            np.full_like(x_grid, sp_val),
            np.full_like(x_grid, sex_val)
        ])
        y_grid = X_grid @ beta_add
        ax.plot(x_grid, y_grid, color=color, linewidth=2.5,
                label=f"{label} fit")

    ax.set_title(f"Additive Model — Sex = {sex_label}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("CL")
    ax.set_ylabel("FL")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

The additive model shifts the line up or down by group, but still forces all groups to share the **same slope**.

Look at the Blue and Orange lines in each panel. They are parallel — that is what the additive model enforces. The only question left is whether this is enough, or whether the species also change the slope.

---


## Part 5: One Slope or Many?

An **interaction term** lets the slope itself differ by group.

We add the product $X \cdot I_{\text{Blue}}$ as a new column:

$$
Y = \beta_0 + \beta_1 X + \beta_2 I_{\text{Blue}}
+ \beta_3 I_{\text{Female}}
+ \beta_4 (X \cdot I_{\text{Blue}}) + Z
$$

The implied slopes are:

- Orange crabs: slope $= \beta_1$
- Blue crabs: slope $= \beta_1 + \beta_4$

If $\beta_4 = 0$, both species share the same slope and the interaction model reduces to the additive model.


### TODO 5a

Build the interaction design matrix. Each row should be:

$$
[1, \; x_i, \; I_{\text{Blue},i}, \; I_{\text{Female},i},
\; x_i \cdot I_{\text{Blue},i}]
$$


In [ ]:
# TODO 5a: Interaction design matrix
x_blue = ...
X_inter = ...

print(f"Shape of X_inter: {X_inter.shape}")

### TODO 5b

Fit the interaction model and compute:

- the coefficient estimates and $R^2$
- the implied Orange slope ($\beta_1$) and Blue slope ($\beta_1 + \beta_4$)


In [ ]:
# TODO 5b: Fit interaction model
model_inter = ...
beta_inter = ...
R2_inter = ...

slope_orange = ...
slope_blue = ...

print("Interaction-model coefficients:")
labels_inter = ["Intercept", "CL", "Blue", "Female", "CL × Blue"]
for name, coef in zip(labels_inter, beta_inter):
    print(f"  {name:>10s}: {coef:+.4f}")
print(f"\nR^2 (interaction): {R2_inter:.4f}")
print(f"Orange slope:      {slope_orange:.4f}")
print(f"Blue slope:        {slope_blue:.4f}")

### TODO 5c

Test whether the interaction term is needed using another nested-model $F$-test:

- reduced model: additive
- full model: interaction

Apply the same $F$-test formula you used in TODO 4c.


In [ ]:
# TODO 5c: F-test for the interaction term
RSS_inter = ...

q_inter = ...
F_interaction = ...
p_value_interaction = ...

print(f"F statistic (additive vs interaction): "
      f"{F_interaction:.3f}")
print(f"p-value:                                "
      f"{p_value_interaction:.3e}")

In [ ]:
# Provided: interaction-model fit visualized by sex
x_grid = np.linspace(x.min(), x.max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for j, sex_label in enumerate(["M", "F"]):
    ax = axes[j]
    sex_val = 1 if sex_label == "F" else 0

    for sp, color, label in [("O", "darkorange", "Orange"),
                             ("B", "dodgerblue", "Blue")]:
        sp_val = 1 if sp == "B" else 0
        mask = (crabs["sex"] == sex_label) & (crabs["sp"] == sp)

        ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                   c=color, s=35, alpha=0.65,
                   edgecolor="white", linewidth=0.3)

        X_grid = np.column_stack([
            np.ones_like(x_grid),
            x_grid,
            np.full_like(x_grid, sp_val),
            np.full_like(x_grid, sex_val),
            x_grid * sp_val
        ])
        y_grid = X_grid @ beta_inter
        ax.plot(x_grid, y_grid, color=color, linewidth=2.5,
                label=f"{label} fit")

    ax.set_title(f"Interaction Model — Sex = {sex_label}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("CL")
    ax.set_ylabel("FL")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

Compare the additive and interaction plots carefully.

In the additive model, the Blue and Orange lines were parallel. In the interaction model, they can cross or diverge. The $F$-test you just computed tells you whether the data justify that additional flexibility.

---


## Part 6: Prediction vs Inference

So far, we have been in the **inference** mindset: which coefficients are significant? Is the richer model justified?

Now we briefly shift to the **prediction** mindset. A more flexible model nearly always improves training-set fit. But does it generalize?

We'll use **5-fold cross-validation** to estimate out-of-sample prediction error for each model.


In [ ]:
# Provided: cross-validation helper
def k_fold_cv_mse(X, y, k=5, seed=42):
    """Compute k-fold cross-validated MSE for OLS.

    Splits the data into k folds, trains on k-1, predicts on
    the held-out fold, and averages the MSE across folds.
    """
    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    folds = np.array_split(indices, k)

    fold_mse = []
    for j in range(k):
        test_idx = folds[j]
        train_idx = np.concatenate(
            [folds[i] for i in range(k) if i != j]
        )
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        model = fit_ols(X_train, y_train)
        y_pred = X_test @ model["beta"]
        fold_mse.append(np.mean((y_test - y_pred)**2))

    return np.mean(fold_mse)

### TODO 6

Compute the 5-fold CV MSE for each model and assemble a comparison table.


In [ ]:
# TODO 6: Cross-validated MSE
cv_mse_pooled = ...
cv_mse_add = ...
cv_mse_inter = ...

comparison_table = pd.DataFrame({
    "model": ["Pooled", "Additive", "Interaction"],
    "parameters": [X_pooled.shape[1],
                   X_add.shape[1],
                   X_inter.shape[1]],
    "train_R2": [R2_pooled, R2_add, R2_inter],
    "cv_mse": [cv_mse_pooled, cv_mse_add, cv_mse_inter],
})

comparison_table.round(4)

Inference and prediction are related but not identical.

A model can be statistically significant without meaningfully improving prediction. Conversely, predictive gains can be modest even when the scientific story is compelling. Both perspectives have value, and a careful analysis considers both.

---


## Part 7: From Regression to a One-Dimensional Score

In Parts 2–5, you studied the *shape* of a relationship and asked how group structure changes it.

Now we'll take a different approach. Instead of predicting one measurement from another, we'll build a **score** that tries to separate the two species.

The idea:

1. Use all five measurements as predictors.
2. Fit a regression with the binary label $G = 1$ (Blue) or $G = 0$ (Orange) as the target.
3. Use the fitted values as a **one-dimensional score**.
4. Test whether the score distributions differ between species.

This pattern — learn a score, visualize it, test it — is exactly what you will do in the final project, at larger scale.


### TODO 7a

Standardize the five measurement columns to have mean 0 and standard deviation 1. Then build a design matrix with an intercept column prepended.

Also create the binary label `g_blue`:
$G_i = 1$ if the $i$th crab is Blue, 0 if Orange.


In [ ]:
# TODO 7a: Standardize features and create the binary label
X_score_raw = crabs[measurement_cols].values

X_score_mean = ...
X_score_std = ...
X_score_stdized = ...

g_blue = ...

# Build design matrix with intercept
X_score = np.column_stack([np.ones(n), X_score_stdized])

print(f"X_score shape: {X_score.shape}")
print(f"Number of Blue crabs:   {int(g_blue.sum())}")
print(f"Number of Orange crabs: {int(n - g_blue.sum())}")

### TODO 7b

Fit a **ridge regression** to predict `g_blue` from the standardized measurements.

Ridge regression modifies the normal equations by adding a penalty to the diagonal:

$$
\hat\beta_{\text{ridge}} =
(X^T X + \lambda I_p^{*})^{-1} X^T y
$$

where $I_p^{*}$ is the identity matrix with a 0 in the top-left corner (so the intercept is **not** penalized).

Use $\lambda = 1.0$ and `np.linalg.solve` to compute $\hat\beta_{\text{ridge}}$.

Then compute the score: $\hat{s} = X \hat\beta_{\text{ridge}}$.


In [ ]:
# TODO 7b: Ridge regression score
lam = 1.0

penalty = np.eye(X_score.shape[1])
penalty[0, 0] = 0.0  # don't penalize the intercept

beta_ridge = ...
score = ...

print("Ridge coefficients:")
score_labels = ["Intercept"] + measurement_cols
for name, coef in zip(score_labels, beta_ridge):
    print(f"  {name:>5s}: {coef:+.4f}")

In [ ]:
# Provided: score histograms by species
fig, ax = plt.subplots(figsize=(8, 5))

score_blue = score[g_blue == 1]
score_orange = score[g_blue == 0]

ax.hist(score_orange, bins=25, alpha=0.55, color="darkorange",
        edgecolor="white", label="Orange")
ax.hist(score_blue, bins=25, alpha=0.55, color="dodgerblue",
        edgecolor="white", label="Blue")

ax.set_xlabel("Ridge score")
ax.set_ylabel("Count")
ax.set_title("A One-Dimensional Score for Species",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### TODO 7c

Use a **Mann-Whitney / Wilcoxon rank-sum test** to test whether Blue crabs tend to have higher scores than Orange crabs.

Use `mannwhitneyu` from `scipy.stats` with `alternative='greater'`.

This tests:

$$
H_0: \text{the score distributions are the same}
$$
$$
H_a: \text{Blue crabs tend to score higher}
$$


In [ ]:
# TODO 7c: Mann-Whitney test on the score
mw_stat, mw_pvalue = ...

print(f"Mann-Whitney statistic: {mw_stat:.2f}")
print(f"p-value:                {mw_pvalue:.3e}")

### Connecting back

Think about the analogy between Parts 2–5 and Part 7:

| Parts 2–5 | Part 7 |
|:---|:---|
| Pooled model ignores species | No score — data is multivariate |
| Build richer models that account for species | Build a score that separates species |
| F-test asks: is the richer model needed? | Mann-Whitney asks: do score distributions differ? |
| Diagnostics reveal hidden group structure | Score histogram visualizes separation |

In the final project, you'll apply this same Part 7 pattern to a much larger dataset, with a more consequential scientific question. The workflow will be the same: build a score, visualize it, test it, improve it.

---


## Report Your Results

Run the cell below and enter the values on the course platform. Round as indicated.


In [ ]:
print("=" * 60)
print("MODULE 5 REPORT VALUES")
print("=" * 60)
print(f"R1.  Pooled slope (4 dec):                     "
      f"{beta1_pooled:.4f}")
print(f"R2.  Pooled R^2 (4 dec):                       "
      f"{R2_pooled:.4f}")
print(f"R3.  Pooled slope t-stat (3 dec):              "
      f"{t_stat_beta1_pooled:.3f}")
print(f"R4.  Mean pooled residual, Blue (4 dec):       "
      f"{mean_resid_by_species.loc['B']:.4f}")
print(f"R5.  Additive R^2 (4 dec):                     "
      f"{R2_add:.4f}")
print(f"R6.  Additive coefficient, Blue (4 dec):       "
      f"{beta_add[2]:.4f}")
print(f"R7.  F-stat pooled vs additive (3 dec):        "
      f"{F_groups:.3f}")
print(f"R8.  Orange slope, interaction model (4 dec):  "
      f"{slope_orange:.4f}")
print(f"R9.  Blue slope, interaction model (4 dec):    "
      f"{slope_blue:.4f}")
print(f"R10. F-stat additive vs interaction (3 dec):   "
      f"{F_interaction:.3f}")
print(f"R11. Mann-Whitney p-value, Part 7 (3 sig fig): "
      f"{mw_pvalue:.3e}")

## Reflection Questions

Discuss these with the course chatbot or on the discussion forum. These are not graded, but thinking through them will prepare you for the final project.


**Q1.** The pooled slope is overwhelmingly significant. Why does a small p-value for the slope not settle the question of whether one common line is adequate?


**Q2.** In the additive model, what does the coefficient on `Blue` represent? What is held fixed when you interpret it?


**Q3.** What is the scientific difference between an additive effect (shifting the line) and an interaction effect (changing the slope)? Can you think of a biological reason why two species might have different FL-vs-CL slopes?


**Q4.** Suppose the $F$-test strongly favors the additive model over the pooled model, but the cross-validated MSE improves only slightly. What does that tell you about the relationship between inference and prediction?


**Q5.** In Part 7, you compressed five measurements into a single score and then tested it. What role does the score play? Why not just test each measurement separately?


**Q6.** How does this lab connect to the broader theme of the course — that pooling can hide important structure? Where in the lab did you first see evidence of this?
